# BPIC-17 Advanced Analysis: Case Duration Prediction

## 1. Objective

This notebook prepares the advanced analysis material for the assignment. It predicts remaining case duration from early-case prefix information in the complete-filtered BPIC-17 log.

The analysis is designed as report material: it compares a simple baseline with a machine-learning model, avoids future leakage, and summarizes predictive usefulness and limitations.


## 2. Imports and Configuration

The notebook uses relative project paths and writes all outputs to `results/` and `figures/advanced_analysis/`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pm4py

warnings.filterwarnings("ignore", message="Install the optional requirement.*")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() or (candidate / "requirements.txt").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "BPI Challenge 2017.xes"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

CASE_ID_COL = "case:concept:name"
ACTIVITY_COL = "concept:name"
TIMESTAMP_COL = "time:timestamp"
LIFECYCLE_COL = "lifecycle:transition"
EVENT_ORIGIN_COL = "EventOrigin"

RANDOM_SEED = 42

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Project root resolved.")
print(f"Data file exists: {DATA_PATH.exists()}")

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ADVANCED_FIGURES_DIR = FIGURES_DIR / "advanced_analysis"
ADVANCED_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PREFIX_EVENTS = 5
TEST_SIZE = 0.30


## 3. Load Complete-Filtered Event Log

The prediction task uses the same complete-event abstraction as process discovery and decision mining.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "BPIC-17 event log not found. Place it at data/BPI Challenge 2017.xes."
    )

raw_log = pm4py.read_xes(str(DATA_PATH), show_progress_bar=False)
df = raw_log if isinstance(raw_log, pd.DataFrame) else pm4py.convert_to_dataframe(raw_log)
df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce", utc=True)

complete_log = (
    df[df[LIFECYCLE_COL].eq("complete")]
    .sort_values([CASE_ID_COL, TIMESTAMP_COL])
    .reset_index(drop=True)
    .copy()
)

print(f"Raw events: {len(df):,}")
print(f"Complete events: {len(complete_log):,}")
print(f"Complete cases: {complete_log[CASE_ID_COL].nunique():,}")
print(f"Complete activities: {complete_log[ACTIVITY_COL].nunique():,}")


## 4. Build Prefix-Level Prediction Dataset

The prediction point is after the first five complete events of a case. Features use only information available up to that prefix. The target is the remaining duration in days after the prefix.


In [ ]:
def save_figure(stem: str, directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(directory / f"{stem}.png", dpi=300)
    plt.savefig(directory / f"{stem}.pdf")
    plt.show()


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()[:60]

complete_log["case_start_time"] = complete_log.groupby(CASE_ID_COL)[TIMESTAMP_COL].transform("min")
complete_log["case_end_time"] = complete_log.groupby(CASE_ID_COL)[TIMESTAMP_COL].transform("max")
complete_log["event_index"] = complete_log.groupby(CASE_ID_COL).cumcount() + 1
complete_log["elapsed_days"] = (complete_log[TIMESTAMP_COL] - complete_log["case_start_time"]).dt.total_seconds() / 86400
complete_log["total_duration_days"] = (complete_log["case_end_time"] - complete_log["case_start_time"]).dt.total_seconds() / 86400

case_lengths = complete_log.groupby(CASE_ID_COL).size().rename("complete_event_count")
eligible_cases = case_lengths[case_lengths >= PREFIX_EVENTS + 1].index
eligible_log = complete_log[complete_log[CASE_ID_COL].isin(eligible_cases)].copy()

prefix_log = eligible_log[eligible_log["event_index"] <= PREFIX_EVENTS].copy()
prefix_end = prefix_log.groupby(CASE_ID_COL).tail(1).copy()

case_attrs = eligible_log[[CASE_ID_COL, "case:LoanGoal", "case:ApplicationType", "case:RequestedAmount"]].drop_duplicates(subset=[CASE_ID_COL])

origin_counts = pd.crosstab(prefix_log[CASE_ID_COL], prefix_log[EVENT_ORIGIN_COL]).add_prefix("prefix_origin_")
top_activities = eligible_log[ACTIVITY_COL].value_counts().head(12).index.tolist()
activity_counts = pd.crosstab(prefix_log[CASE_ID_COL], prefix_log[ACTIVITY_COL])
activity_counts = activity_counts.reindex(columns=top_activities, fill_value=0)
activity_counts.columns = [f"prefix_activity_{safe_name(col)}" for col in activity_counts.columns]

prediction_data = (
    prefix_end[[CASE_ID_COL, "elapsed_days", "total_duration_days", ACTIVITY_COL]]
    .rename(columns={ACTIVITY_COL: "last_prefix_activity"})
    .merge(case_attrs, on=CASE_ID_COL, how="left")
    .merge(origin_counts, left_on=CASE_ID_COL, right_index=True, how="left")
    .merge(activity_counts, left_on=CASE_ID_COL, right_index=True, how="left")
)
prediction_data = prediction_data.fillna(0)
prediction_data["remaining_duration_days"] = prediction_data["total_duration_days"] - prediction_data["elapsed_days"]
prediction_data = prediction_data[prediction_data["remaining_duration_days"] >= 0].copy()

prediction_data.to_csv(RESULTS_DIR / "advanced_duration_prediction_dataset_overview.csv", index=False)

print(f"Eligible cases: {len(prediction_data):,}")
display(prediction_data[["elapsed_days", "total_duration_days", "remaining_duration_days"]].describe())


## 5. Train Baseline and Random Forest Models

The baseline predicts the median remaining duration from the training set. The random forest uses case attributes and prefix behavior. The evaluation uses a fixed train/test split for reproducibility.


In [ ]:
target_col = "remaining_duration_days"
exclude_cols = [CASE_ID_COL, "total_duration_days", target_col]
feature_cols = [col for col in prediction_data.columns if col not in exclude_cols]

numeric_features = [
    col for col in feature_cols
    if pd.api.types.is_numeric_dtype(prediction_data[col])
]
categorical_features = [col for col in feature_cols if col not in numeric_features]

X = prediction_data[feature_cols]
y = prediction_data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

baseline_prediction = np.full(shape=len(y_test), fill_value=float(y_train.median()))

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

rf_model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=14,
                min_samples_leaf=20,
                n_jobs=-1,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)
rf_model.fit(X_train, y_train)
rf_prediction = rf_model.predict(X_test)


def regression_metrics(model_name: str, y_true: pd.Series, y_pred: np.ndarray) -> dict:
    return {
        "model": model_name,
        "mae_days": mean_absolute_error(y_true, y_pred),
        "rmse_days": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": r2_score(y_true, y_pred),
        "train_cases": len(y_train),
        "test_cases": len(y_true),
        "prediction_target": target_col,
        "prefix_events": PREFIX_EVENTS,
    }

advanced_metrics = pd.DataFrame(
    [
        regression_metrics("MedianBaseline", y_test, baseline_prediction),
        regression_metrics("RandomForestRegressor", y_test, rf_prediction),
    ]
)
advanced_metrics.to_csv(RESULTS_DIR / "advanced_analysis_metrics.csv", index=False)
display(advanced_metrics)


## 6. Visualize Prediction Quality

The plots compare predicted and actual remaining duration, inspect the error distribution, and show the most important features used by the random forest.


In [ ]:
plot_data = pd.DataFrame(
    {
        "actual_remaining_days": y_test.to_numpy(),
        "predicted_remaining_days": rf_prediction,
        "error_days": rf_prediction - y_test.to_numpy(),
    }
)
plot_data.to_csv(RESULTS_DIR / "advanced_analysis_predictions.csv", index=False)

plt.figure(figsize=(8, 8))
plt.scatter(plot_data["actual_remaining_days"], plot_data["predicted_remaining_days"], alpha=0.25, s=12)
limit = np.nanpercentile(plot_data[["actual_remaining_days", "predicted_remaining_days"]].to_numpy(), 99)
plt.plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
plt.xlim(0, limit)
plt.ylim(0, limit)
plt.title("Predicted vs Actual Remaining Case Duration")
plt.xlabel("Actual Remaining Duration (days)")
plt.ylabel("Predicted Remaining Duration (days)")
save_figure("predicted_vs_actual_duration", ADVANCED_FIGURES_DIR)

plt.figure(figsize=(10, 6))
plt.hist(plot_data["error_days"], bins=60)
plt.title("Prediction Error Distribution")
plt.xlabel("Prediction Error (predicted - actual, days)")
plt.ylabel("Number of Cases")
save_figure("error_distribution", ADVANCED_FIGURES_DIR)

feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()
feature_importance = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "importance": rf_model.named_steps["model"].feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .head(25)
)
feature_importance.to_csv(RESULTS_DIR / "advanced_analysis_feature_importance.csv", index=False)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance["feature"][::-1], feature_importance["importance"][::-1])
plt.title("Random Forest Feature Importance for Remaining Duration Prediction")
plt.xlabel("Importance")
plt.ylabel("Feature")
save_figure("feature_importance", ADVANCED_FIGURES_DIR)

display(feature_importance)


## 7. Save Advanced Analysis Summary

The summary is intermediate report material. It records the task definition, model comparison, generated files, and key limitations.


In [ ]:
summary_lines = [
    "# Advanced Analysis Summary: Case Duration Prediction",
    "",
    "## Scope",
    "",
    f"This advanced analysis predicts remaining case duration after the first {PREFIX_EVENTS} complete events of a case. The task uses only case attributes and prefix-level behavior observed before the prediction point.",
    "",
    "## Dataset",
    "",
    f"- Eligible cases: {len(prediction_data):,}",
    f"- Train cases: {len(y_train):,}",
    f"- Test cases: {len(y_test):,}",
    f"- Target: `{target_col}`",
    "",
    "## Model Metrics",
    "",
]

columns = ["model", "mae_days", "rmse_days", "r2", "train_cases", "test_cases"]
summary_lines.append("| " + " | ".join(columns) + " |")
summary_lines.append("| " + " | ".join(["---"] * len(columns)) + " |")
for _, row in advanced_metrics[columns].iterrows():
    values = []
    for col in columns:
        value = row[col]
        values.append(f"{value:.4f}" if isinstance(value, float) else str(value))
    summary_lines.append("| " + " | ".join(values) + " |")

summary_lines.extend([
    "",
    "## Interpretation Notes",
    "",
    "- The median baseline is included to show whether the machine-learning model adds predictive value.",
    "- The random forest captures non-linear relationships in early case behavior but is less interpretable than a shallow tree.",
    "- The target is remaining duration after a fixed prefix, so it avoids using future events as features.",
    "- Errors may be larger for unusually long-running cases, which should be discussed as a limitation.",
    "",
    "## Generated Files",
    "",
    "- `results/advanced_duration_prediction_dataset_overview.csv`",
    "- `results/advanced_analysis_metrics.csv`",
    "- `results/advanced_analysis_predictions.csv`",
    "- `results/advanced_analysis_feature_importance.csv`",
    "- `figures/advanced_analysis/predicted_vs_actual_duration.pdf/png`",
    "- `figures/advanced_analysis/error_distribution.pdf/png`",
    "- `figures/advanced_analysis/feature_importance.pdf/png`",
])

(RESULTS_DIR / "advanced_analysis_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")
print("Saved advanced analysis summary.")


## 8. Report Notes

This notebook completes the advanced analysis material. The most useful report artifacts are the baseline-vs-random-forest metrics, predicted-vs-actual plot, error distribution, and feature importance interpretation.
